# PHYT1D — Module 05: MCMC Window A — Core Parameter Identification

Identifies 8 core glucose-insulin parameters from **meal-only CGM segments**
(no exercise, ≥ 4 h from prior bolus, ≥ 6 h windows).

**Algorithm:** Adaptive Single-Component Metropolis-Hastings (SCMH)  
**Posterior fit:** t-copula on 1000 retained samples  
**Pass criterion:** MARD < 10% (Cappon 2023)

θ_A = {SI, SG, Gb, p2, kd, ka2, kempt, kabs}


In [ ]:
import numpy as np
from scipy import stats


## 5.1 Log-Prior

In [ ]:
def log_prior_A(theta, p_dict=None):
    """
    Log prior for Group A parameters.

    Parameters
    ----------
    theta  : dict — {SI, SG, Gb, p2, kd, ka2, kempt, kabs}
    p_dict : dict — GROUP_A_PARAMS (imported or inline)

    Returns
    -------
    float — log p(θ)
    """
    # Prior definitions (matching 04_parameters.py)
    priors = {
        "SI"   : ("lognormal", -5.3,  0.6),
        "SG"   : ("normal",    0.025, 0.013),
        "Gb"   : ("normal",    120.0, 10.0),
        "p2"   : ("normal",    0.05,  0.02),
        "kd"   : ("normal",    0.026, 0.013),
        "ka2"  : ("normal",    0.066, 0.030),
        "kempt": ("normal",    0.055, 0.020),
        "kabs" : ("normal",    0.057, 0.020),
    }
    lp = 0.0
    for name, val in theta.items():
        if val <= 0:
            return -np.inf
        kind = priors[name][0]
        if kind == "lognormal":
            mu, sigma = priors[name][1], priors[name][2]
            lp += stats.norm.logpdf(np.log(val), mu, sigma) - np.log(val)
        else:
            mu, sigma = priors[name][1], priors[name][2]
            lp += stats.norm.logpdf(val, mu, sigma)
    return lp


## 5.2 Log-Likelihood — Gaussian on CGM Residuals

In [ ]:
def log_likelihood_A(theta, cgm_times, cgm_obs, meal_events, insulin_events,
                     sigma_cgm=5.0, fixed=None):
    """
    Gaussian log-likelihood on CGM residuals.

    L(Y | θ) = -0.5 * Σ [(IG_obs(t) - IG_sim(t))² / σ²]

    Parameters
    ----------
    theta         : dict — Group A parameters
    cgm_times     : array [min] — CGM measurement times
    cgm_obs       : array [mg/dL] — observed CGM values
    meal_events   : list of (time_min, CHO_g_per_kg)
    insulin_events: list of (time_min, dose_mU_per_kg)
    sigma_cgm     : float [mg/dL] — CGM noise standard deviation
    fixed         : dict — Group D fixed parameters

    Returns
    -------
    float — log L(Y | θ, U)
    """
    from 06_simulator import run_ode_window  # forward model
    try:
        ig_sim = run_ode_window(theta, cgm_times, meal_events, insulin_events,
                                fixed=fixed)
        residuals = cgm_obs - ig_sim
        ll = -0.5 * np.sum((residuals / sigma_cgm) ** 2)
        ll -= len(cgm_obs) * np.log(sigma_cgm * np.sqrt(2 * np.pi))
        return ll
    except Exception:
        return -np.inf


## 5.3 Adaptive SCMH Sampler

In [ ]:
def adaptive_SCMH(theta_init, log_posterior_fn, n_samples=2000, burn_in=1000,
                  adapt_interval=50, target_rate=0.23, rng=None, verbose=True):
    """
    Adaptive Single-Component Metropolis-Hastings (SCMH).

    Updates one parameter at a time; adapts proposal σ to target acceptance rate.

    Parameters
    ----------
    theta_init       : dict   — initial parameter values
    log_posterior_fn : callable(dict) → float
    n_samples        : int    — total iterations (including burn-in)
    burn_in          : int    — samples discarded (first burn_in)
    adapt_interval   : int    — adaptation frequency [iterations]
    target_rate      : float  — target acceptance rate (Gelman: 0.23 for high-dim)
    rng              : np.random.Generator

    Returns
    -------
    chain  : list of dicts — post-burn-in samples
    accept : dict          — per-parameter acceptance rates
    """
    if rng is None:
        rng = np.random.default_rng(42)

    param_names = list(theta_init.keys())
    theta_cur   = {k: v for k, v in theta_init.items()}
    lp_cur      = log_posterior_fn(theta_cur)

    # Adaptive proposal σ (initialised to 10% of prior means)
    sigma_prop  = {k: abs(v) * 0.10 + 1e-6 for k, v in theta_cur.items()}
    accept_cnt  = {k: 0 for k in param_names}
    total_cnt   = {k: 0 for k in param_names}

    chain = []

    for i in range(n_samples):
        for name in param_names:
            # Propose new value for this parameter
            theta_prop = {k: v for k, v in theta_cur.items()}
            theta_prop[name] = theta_cur[name] + rng.normal(0, sigma_prop[name])

            if theta_prop[name] <= 0:
                total_cnt[name] += 1
                continue

            lp_prop = log_posterior_fn(theta_prop)
            log_alpha = lp_prop - lp_cur

            if np.log(rng.uniform()) < log_alpha:
                theta_cur = theta_prop
                lp_cur    = lp_prop
                accept_cnt[name] += 1
            total_cnt[name] += 1

        # Adapt proposal σ every adapt_interval steps (during burn-in)
        if i < burn_in and (i + 1) % adapt_interval == 0:
            for name in param_names:
                rate = accept_cnt[name] / max(total_cnt[name], 1)
                if rate > target_rate * 1.2:
                    sigma_prop[name] *= 1.1
                elif rate < target_rate * 0.8:
                    sigma_prop[name] *= 0.9

        if i >= burn_in:
            chain.append({k: v for k, v in theta_cur.items()})

        if verbose and (i + 1) % 500 == 0:
            rate = {k: accept_cnt[k] / max(total_cnt[k], 1) for k in param_names}
            avg  = np.mean(list(rate.values()))
            print(f"  Iteration {i+1:>5d}/{n_samples}  avg accept={avg:.2f}  lp={lp_cur:.2f}")

    accept_rates = {k: accept_cnt[k] / max(total_cnt[k], 1) for k in param_names}
    return chain, accept_rates


## 5.4 t-Copula Posterior Fit

In [ ]:
def fit_tcopula_posterior(chain, nu=4):
    """
    Fit a t-copula to the MCMC chain for smooth posterior density estimation.
    Returns 1000 posterior realisations (Demarta & McNeil 2007).

    Parameters
    ----------
    chain : list of dicts — MCMC posterior samples
    nu    : int — degrees of freedom for the t-copula

    Returns
    -------
    realisations : np.ndarray shape (1000, n_params)
    param_names  : list of str
    """
    from scipy.stats import t as t_dist, norm as norm_dist

    param_names = list(chain[0].keys())
    X = np.array([[s[k] for k in param_names] for s in chain])   # (n_samples, n_params)
    n, d = X.shape

    # Step 1: Convert to pseudo-observations U ∈ (0,1)
    U = np.zeros_like(X)
    for j in range(d):
        ranks = stats.rankdata(X[:, j])
        U[:, j] = ranks / (n + 1)

    # Step 2: Convert to t-scores
    T = t_dist.ppf(U, df=nu)

    # Step 3: Estimate covariance of t-scores
    cov  = np.cov(T.T)
    mean = np.mean(T, axis=0)

    # Step 4: Sample from multivariate-t with estimated covariance
    rng   = np.random.default_rng(0)
    chi2  = rng.chisquare(nu, size=1000)
    gauss = rng.multivariate_normal(np.zeros(d), cov, size=1000)
    T_new = mean + gauss / np.sqrt(chi2[:, None] / nu)

    # Step 5: Back-transform through marginal CDFs
    U_new = t_dist.cdf(T_new, df=nu)
    X_new = np.zeros_like(T_new)
    for j in range(d):
        X_new[:, j] = np.quantile(X[:, j], np.clip(U_new[:, j], 1e-6, 1-1e-6))

    return X_new, param_names


## 5.5 Identifiability Check

In [ ]:
def check_identifiability(chain, prior_stds):
    """
    Identifiability check: posterior SD < 3× prior SD per parameter.
    (Pillonetto 2003)

    Parameters
    ----------
    chain      : list of dicts — MCMC posterior samples
    prior_stds : dict          — {param: prior_std}

    Returns
    -------
    report : dict — {param: (post_std, 3*prior_std, identified?)}
    """
    param_names = list(chain[0].keys())
    samples = {k: [s[k] for s in chain] for k in param_names}
    report  = {}
    print("\n── Identifiability Check (Pillonetto 2003) ──────────────────")
    print(f"  {'Param':8s}  {'Post SD':>10s}  {'3× Prior SD':>12s}  {'OK?':>5s}")
    for k in param_names:
        post_sd  = np.std(samples[k])
        thresh   = 3.0 * prior_stds.get(k, np.inf)
        ok       = post_sd < thresh
        report[k] = (post_sd, thresh, ok)
        print(f"  {k:8s}  {post_sd:10.6f}  {thresh:12.6f}  {'✓' if ok else '✗'}")
    return report


## 5.6 Window A Evaluation — MARD

In [ ]:
def compute_MARD(ig_true, ig_sim):
    """
    MARD = (1 / (100·D)) · Σ_p Σ_k |IG_obs - IG_sim| / IG_obs

    For a single patient: MARD = mean(|IG_obs - IG_sim| / IG_obs) × 100 [%]
    Pass threshold: MARD < 10% (Cappon 2023).

    Parameters
    ----------
    ig_true : array — ground-truth IG [mg/dL]
    ig_sim  : array — simulated IG [mg/dL]

    Returns
    -------
    float — MARD [%]
    """
    return float(np.mean(np.abs(ig_true - ig_sim) / ig_true) * 100.0)


def evaluate_window_A(chain, ig_true, ig_sim_fn, pass_threshold=10.0):
    """
    Run MARD evaluation for Window A.
    Returns pass/fail and metric values.
    """
    # Median posterior prediction
    param_names = list(chain[0].keys())
    medians = {k: np.median([s[k] for s in chain]) for k in param_names}
    ig_sim  = ig_sim_fn(medians)
    mard    = compute_MARD(ig_true, ig_sim)
    passed  = mard < pass_threshold

    print(f"\n── Window A Evaluation ─────────────────────────────────────")
    print(f"  MARD = {mard:.2f}%   threshold = {pass_threshold}%   {'PASS ✓' if passed else 'FAIL ✗'}")
    return {"MARD": mard, "passed": passed, "theta_median": medians}
